In [1]:
# ============================================================
# 06D.1 — LOAD 06C PCA RESULTS
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

print("=" * 70)
print("06D — PCA LOADINGS & CONTRIBUTIONS")
print("=" * 70)


# ------------------------------------------------------------
# 1. Locate project and 06C artifact directory
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parents[1]

PCA_ARTIFACT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "pca_analysis"
)

assert PCA_ARTIFACT_DIR.exists(), (
    f"06C PCA artifact directory not found:\n"
    f"{PCA_ARTIFACT_DIR}"
)

print("\n06C artifact directory:")
print(PCA_ARTIFACT_DIR)


# ------------------------------------------------------------
# 2. Representation names
# ------------------------------------------------------------

REPRESENTATION_NAMES = [
    "baseline",
    "distribution_aware",
    "yeo_johnson",
    "block_balanced",
    "ndsi_derived_excluded",
    "ndsi_family_excluded",
]


# ------------------------------------------------------------
# 3. Load PCA results
# ------------------------------------------------------------

PCA_06D = {}

for name in REPRESENTATION_NAMES:

    # PCA components
    components = np.load(
        PCA_ARTIFACT_DIR
        / f"{name}_components.npy"
    )

    # Explained variance
    explained_variance = np.load(
        PCA_ARTIFACT_DIR
        / f"{name}_explained_variance.npy"
    )

    # Explained variance ratio
    explained_variance_ratio = np.load(
        PCA_ARTIFACT_DIR
        / f"{name}_explained_variance_ratio.npy"
    )

    # Singular values
    singular_values = np.load(
        PCA_ARTIFACT_DIR
        / f"{name}_singular_values.npy"
    )

    # Feature names
    with open(
        PCA_ARTIFACT_DIR
        / f"{name}_feature_names.json",
        "r",
        encoding="utf-8"
    ) as f:
        feature_names = json.load(f)

    # Parallel-analysis retained PCs
    retained = np.load(
        PCA_ARTIFACT_DIR
        / f"{name}_pa_retained.npy"
    )

    PCA_06D[name] = {
        "components": components,
        "explained_variance": explained_variance,
        "explained_variance_ratio": explained_variance_ratio,
        "singular_values": singular_values,
        "feature_names": feature_names,
        "retained": retained,
    }


# ------------------------------------------------------------
# 4. Validate dimensions
# ------------------------------------------------------------

for name in REPRESENTATION_NAMES:

    result = PCA_06D[name]

    components = result["components"]
    eigenvalues = result["explained_variance"]
    evr = result["explained_variance_ratio"]
    features = result["feature_names"]
    retained = result["retained"]

    n_features = len(features)

    assert components.shape == (
        n_features,
        n_features
    ), (
        f"{name}: component matrix dimension mismatch"
    )

    assert len(eigenvalues) == n_features, (
        f"{name}: eigenvalue count mismatch"
    )

    assert len(evr) == n_features, (
        f"{name}: explained variance count mismatch"
    )

    assert len(retained) == n_features, (
        f"{name}: retained-component mask mismatch"
    )

    assert np.isfinite(components).all(), (
        f"{name}: non-finite components"
    )

    assert np.isfinite(eigenvalues).all(), (
        f"{name}: non-finite eigenvalues"
    )

    assert np.isfinite(evr).all(), (
        f"{name}: non-finite explained variance"
    )


# ------------------------------------------------------------
# 5. Build validation summary
# ------------------------------------------------------------

validation_rows = []

for name in REPRESENTATION_NAMES:

    result = PCA_06D[name]

    validation_rows.append({
        "representation": name,
        "n_features": len(result["feature_names"]),
        "n_PCs": len(result["explained_variance"]),
        "retained_PCs": int(
            np.sum(result["retained"])
        ),
        "components_finite": np.isfinite(
            result["components"]
        ).all(),
        "eigenvalues_finite": np.isfinite(
            result["explained_variance"]
        ).all(),
        "feature_names_loaded": len(
            result["feature_names"]
        )
    })


loadings_handoff_summary = pd.DataFrame(
    validation_rows
)


# ------------------------------------------------------------
# 6. Display
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("06C → 06D HANDOFF")
print("=" * 70)

display(loadings_handoff_summary)


print("\n" + "=" * 70)
print("06D.1 VALIDATION PASSED")
print("=" * 70)

print("""
✓ PCA components loaded
✓ Eigenvalues loaded
✓ Explained variance loaded
✓ Feature names loaded
✓ Parallel-analysis decisions loaded
✓ Dimensions verified
✓ No non-finite PCA quantities

Ready to calculate corrected PCA loadings.
""")

06D — PCA LOADINGS & CONTRIBUTIONS

06C artifact directory:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_analysis

06C → 06D HANDOFF


,representation,n_features,n_PCs,retained_PCs,components_finite,eigenvalues_finite,feature_names_loaded
0,baseline,50,50,9,True,True,50
1,distribution_aware,50,50,9,True,True,50
2,yeo_johnson,50,50,9,True,True,50
3,block_balanced,50,50,7,True,True,50
4,ndsi_derived_excluded,48,48,9,True,True,48
5,ndsi_family_excluded,46,46,9,True,True,46



06D.1 VALIDATION PASSED

✓ PCA components loaded
✓ Eigenvalues loaded
✓ Explained variance loaded
✓ Feature names loaded
✓ Parallel-analysis decisions loaded
✓ Dimensions verified
✓ No non-finite PCA quantities

Ready to calculate corrected PCA loadings.



In [2]:
# ============================================================
# 06D.2 — PCA LOADINGS & FEATURE CONTRIBUTIONS
# ============================================================

print("=" * 70)
print("06D.2 — PCA LOADINGS & FEATURE CONTRIBUTIONS")
print("=" * 70)


# ------------------------------------------------------------
# 1. Calculate loadings
# ------------------------------------------------------------
#
# sklearn PCA components_ are unit eigenvectors.
#
# Conventional PCA loadings:
#
#       loading_jk = component_jk * sqrt(eigenvalue_k)
#
# Rows    = features
# Columns = principal components
#
# ------------------------------------------------------------

for name in REPRESENTATION_NAMES:

    result = PCA_06D[name]

    components = result["components"]
    eigenvalues = result["explained_variance"]

    loadings = (
        components.T
        * np.sqrt(eigenvalues)
    )

    result["loadings"] = loadings


# ------------------------------------------------------------
# 2. Calculate squared loadings
# ------------------------------------------------------------
#
# Squared loadings quantify the amount of variance in a
# standardized feature associated with each component.
#
# ------------------------------------------------------------

for name in REPRESENTATION_NAMES:

    loadings = PCA_06D[name]["loadings"]

    squared_loadings = loadings ** 2

    PCA_06D[name]["squared_loadings"] = squared_loadings


# ------------------------------------------------------------
# 3. Calculate within-PC relative contributions
# ------------------------------------------------------------
#
# For each PC:
#
# contribution = squared loading /
#                sum of squared loadings across features
#
# This lets us rank which variables contribute most strongly
# to each retained component.
# ------------------------------------------------------------

for name in REPRESENTATION_NAMES:

    squared_loadings = (
        PCA_06D[name]["squared_loadings"]
    )

    column_totals = squared_loadings.sum(
        axis=0,
        keepdims=True
    )

    contributions = (
        squared_loadings
        / column_totals
    )

    PCA_06D[name]["contributions"] = contributions


# ------------------------------------------------------------
# 4. Validate loading calculations
# ------------------------------------------------------------

for name in REPRESENTATION_NAMES:

    result = PCA_06D[name]

    loadings = result["loadings"]
    squared = result["squared_loadings"]
    contributions = result["contributions"]

    n_features = len(
        result["feature_names"]
    )

    n_pcs = len(
        result["explained_variance"]
    )

    assert loadings.shape == (
        n_features,
        n_pcs
    )

    assert squared.shape == (
        n_features,
        n_pcs
    )

    assert contributions.shape == (
        n_features,
        n_pcs
    )

    assert np.isfinite(loadings).all()
    assert np.isfinite(squared).all()
    assert np.isfinite(contributions).all()

    # Relative contributions within each PC
    # must sum to 1.
    contribution_totals = (
        contributions.sum(axis=0)
    )

    assert np.allclose(
        contribution_totals,
        1.0,
        atol=1e-10
    ), (
        f"{name}: contribution totals do not equal 1"
    )


# ------------------------------------------------------------
# 5. Verify loading magnitude against explained variance
# ------------------------------------------------------------
#
# Because the input features were standardized before PCA,
# the sum of squared loadings for PC k should equal its
# eigenvalue.
# ------------------------------------------------------------

loading_variance_rows = []

for name in REPRESENTATION_NAMES:

    result = PCA_06D[name]

    squared = result["squared_loadings"]
    eigenvalues = result["explained_variance"]

    reconstructed_eigenvalues = (
        squared.sum(axis=0)
    )

    max_error = np.max(
        np.abs(
            reconstructed_eigenvalues
            - eigenvalues
        )
    )

    loading_variance_rows.append({
        "representation": name,
        "max_eigenvalue_reconstruction_error": max_error
    })

loading_variance_validation = pd.DataFrame(
    loading_variance_rows
)


# ------------------------------------------------------------
# 6. Display validation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LOADING / EIGENVALUE VALIDATION")
print("=" * 70)

display(
    loading_variance_validation
)


# ------------------------------------------------------------
# 7. Build long-format feature loading table
# ------------------------------------------------------------

loading_rows = []

for name in REPRESENTATION_NAMES:

    result = PCA_06D[name]

    features = result["feature_names"]
    loadings = result["loadings"]
    contributions = result["contributions"]

    for j, feature in enumerate(features):

        for k in range(loadings.shape[1]):

            loading_rows.append({
                "representation": name,
                "feature": feature,
                "PC": k + 1,
                "loading": loadings[j, k],
                "abs_loading": abs(
                    loadings[j, k]
                ),
                "squared_loading": (
                    loadings[j, k] ** 2
                ),
                "contribution": (
                    contributions[j, k]
                )
            })


pca_loading_table = pd.DataFrame(
    loading_rows
)


# ------------------------------------------------------------
# 8. Identify dominant features for retained PCs
# ------------------------------------------------------------

dominant_rows = []

for name in REPRESENTATION_NAMES:

    result = PCA_06D[name]

    retained = result["retained"]
    features = result["feature_names"]
    loadings = result["loadings"]
    contributions = result["contributions"]

    retained_pcs = np.where(
        retained
    )[0]

    for k in retained_pcs:

        ranking = np.argsort(
            np.abs(loadings[:, k])
        )[::-1]

        # Top 10 features by absolute loading
        top_indices = ranking[:10]

        for rank, j in enumerate(
            top_indices,
            start=1
        ):

            dominant_rows.append({
                "representation": name,
                "PC": k + 1,
                "rank": rank,
                "feature": features[j],
                "loading": loadings[j, k],
                "abs_loading": abs(
                    loadings[j, k]
                ),
                "contribution_pct": (
                    contributions[j, k] * 100
                )
            })


dominant_loading_table = pd.DataFrame(
    dominant_rows
)


# ------------------------------------------------------------
# 9. Show top features for the first retained PCs
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TOP FEATURES — RETAINED PRINCIPAL COMPONENTS")
print("=" * 70)

for name in REPRESENTATION_NAMES:

    retained = np.where(
        PCA_06D[name]["retained"]
    )[0]

    print("\n" + "-" * 70)
    print(name)

    # Show first three retained PCs here.
    for pc_index in retained[:3]:

        subset = dominant_loading_table[
            (
                dominant_loading_table[
                    "representation"
                ] == name
            )
            &
            (
                dominant_loading_table["PC"]
                == pc_index + 1
            )
        ]

        print(
            f"\nPC{pc_index + 1}:"
        )

        display(
            subset[
                [
                    "rank",
                    "feature",
                    "loading",
                    "contribution_pct"
                ]
            ].round(4)
        )


# ------------------------------------------------------------
# 10. Final validation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("06D.2 VALIDATION PASSED")
print("=" * 70)

print("""
✓ Conventional PCA loadings calculated
✓ Squared loadings calculated
✓ Relative feature contributions calculated
✓ Contributions sum to 100% within every PC
✓ Eigenvalues reconstructed from squared loadings
✓ Dominant features identified for retained PCs
""")

06D.2 — PCA LOADINGS & FEATURE CONTRIBUTIONS

LOADING / EIGENVALUE VALIDATION


,representation,max_eigenvalue_reconstruction_error
0,baseline,9.769963e-15
1,distribution_aware,7.549517e-15
2,yeo_johnson,1.776357e-14
3,block_balanced,6.661338e-16
4,ndsi_derived_excluded,1.598721e-14
5,ndsi_family_excluded,2.486900e-14



TOP FEATURES — RETAINED PRINCIPAL COMPONENTS

----------------------------------------------------------------------
baseline

PC1:


,rank,feature,loading,contribution_pct
0,1,mfcc_09_within_std_mean,0.8777,7.3258
1,2,mfcc_08_within_std_mean,0.8696,7.1910
2,3,mfcc_12_within_std_mean,0.8683,7.1703
3,4,mfcc_10_within_std_mean,0.8590,7.0174
4,5,mfcc_06_within_std_mean,0.8356,6.6403
5,6,mfcc_05_within_std_mean,0.8324,6.5900
6,7,mfcc_07_within_std_mean,0.8301,6.5524
7,8,mfcc_11_within_std_mean,0.8274,6.5099
8,9,mfcc_13_within_std_mean,0.8003,6.0913
9,10,mfcc_03_within_std_mean,0.6921,4.5550



PC2:


...


PC2:


,rank,feature,loading,contribution_pct
440,1,spectral_centroid_mean_mean,0.8299,12.2620
441,2,mfcc_02_mean,-0.8235,12.0731
442,3,spectral_rolloff_mean_mean,0.8174,11.8939
443,4,zcr_mean_mean,0.7568,10.1959
444,5,spectral_entropy_eps_mean,-0.5868,6.1306
445,6,zcr_std_mean,0.5718,5.8212
446,7,spectral_bandwidth_mean_mean,0.5716,5.8154
447,8,adi_mean,0.5167,4.7532
448,9,spectral_entropy_eas_mean,-0.5152,4.7257
449,10,spectral_centroid_std_mean,0.4988,4.4298



PC3:


,rank,feature,loading,contribution_pct
450,1,spectral_entropy_ecu_mean,0.5485,9.2774
451,2,spectral_entropy_ecv_mean,0.5256,8.5194
452,3,mfcc_03_mean,0.5184,8.2884
453,4,spectral_entropy_eas_mean,0.4826,7.1810
454,5,mfcc_01_mean,-0.4581,6.4705
455,6,spectral_bandwidth_mean_mean,0.4401,5.9741
456,7,spectral_entropy_eps_skew_mean,0.4352,5.8404
457,8,bioacoustic_index_mean,-0.4116,5.2238
458,9,spectral_entropy_eps_kurt_mean,0.4105,5.1956
459,10,spectral_rolloff_std_mean,0.3705,4.2340



06D.2 VALIDATION PASSED

✓ Conventional PCA loadings calculated
✓ Squared loadings calculated
✓ Relative feature contributions calculated
✓ Contributions sum to 100% within every PC
✓ Eigenvalues reconstructed from squared loadings
✓ Dominant features identified for retained PCs



In [3]:
# ============================================================
# 06D.3 — CROSS-REPRESENTATION LOADING STABILITY
# ============================================================

from itertools import combinations

print("=" * 70)
print("06D.3 — CROSS-REPRESENTATION LOADING STABILITY")
print("=" * 70)


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

N_COMPARE_PCS = 3
TOP_K = 10


# ------------------------------------------------------------
# Helper: extract loadings for shared features
# ------------------------------------------------------------

def shared_loading_vectors(
    representation_a,
    representation_b,
    pc_index
):
    """
    Return loading vectors restricted to features shared by
    two representations.
    """

    result_a = PCA_06D[representation_a]
    result_b = PCA_06D[representation_b]

    features_a = result_a["feature_names"]
    features_b = result_b["feature_names"]

    shared_features = sorted(
        set(features_a).intersection(features_b)
    )

    index_a = {
        feature: i
        for i, feature in enumerate(features_a)
    }

    index_b = {
        feature: i
        for i, feature in enumerate(features_b)
    }

    vector_a = np.array([
        result_a["loadings"][
            index_a[feature],
            pc_index
        ]
        for feature in shared_features
    ])

    vector_b = np.array([
        result_b["loadings"][
            index_b[feature],
            pc_index
        ]
        for feature in shared_features
    ])

    return (
        np.array(shared_features),
        vector_a,
        vector_b
    )


# ------------------------------------------------------------
# 1. Pairwise loading correlations
# ------------------------------------------------------------

pairwise_rows = []

for rep_a, rep_b in combinations(
    REPRESENTATION_NAMES,
    2
):

    for pc_index in range(
        N_COMPARE_PCS
    ):

        features, loading_a, loading_b = (
            shared_loading_vectors(
                rep_a,
                rep_b,
                pc_index
            )
        )

        # PCA signs are arbitrary.
        # First calculate raw correlation.
        raw_corr = np.corrcoef(
            loading_a,
            loading_b
        )[0, 1]

        # Sign-align representation B to A.
        sign = (
            1
            if np.dot(
                loading_a,
                loading_b
            ) >= 0
            else -1
        )

        aligned_b = loading_b * sign

        aligned_corr = np.corrcoef(
            loading_a,
            aligned_b
        )[0, 1]

        # Compare magnitudes independently of sign.
        absolute_corr = np.corrcoef(
            np.abs(loading_a),
            np.abs(loading_b)
        )[0, 1]

        pairwise_rows.append({
            "representation_a": rep_a,
            "representation_b": rep_b,
            "PC": pc_index + 1,
            "n_shared_features": len(features),
            "raw_loading_corr": raw_corr,
            "sign_aligned_loading_corr": aligned_corr,
            "absolute_loading_corr": absolute_corr,
            "sign_alignment": sign
        })


pairwise_loading_stability = pd.DataFrame(
    pairwise_rows
)


# ------------------------------------------------------------
# 2. Display pairwise correlations
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PAIRWISE LOADING CORRELATIONS")
print("=" * 70)

display(
    pairwise_loading_stability.round(3)
)


# ------------------------------------------------------------
# 3. Top-feature overlap
# ------------------------------------------------------------

overlap_rows = []

for rep_a, rep_b in combinations(
    REPRESENTATION_NAMES,
    2
):

    for pc_index in range(
        N_COMPARE_PCS
    ):

        result_a = PCA_06D[rep_a]
        result_b = PCA_06D[rep_b]

        features_a = np.array(
            result_a["feature_names"]
        )

        features_b = np.array(
            result_b["feature_names"]
        )

        abs_a = np.abs(
            result_a["loadings"][
                :,
                pc_index
            ]
        )

        abs_b = np.abs(
            result_b["loadings"][
                :,
                pc_index
            ]
        )

        top_a = set(
            features_a[
                np.argsort(abs_a)[::-1][:TOP_K]
            ]
        )

        top_b = set(
            features_b[
                np.argsort(abs_b)[::-1][:TOP_K]
            ]
        )

        shared_top = (
            top_a.intersection(top_b)
        )

        overlap_rows.append({
            "representation_a": rep_a,
            "representation_b": rep_b,
            "PC": pc_index + 1,
            "top_k": TOP_K,
            "shared_top_features": len(shared_top),
            "overlap_fraction": (
                len(shared_top) / TOP_K
            ),
            "shared_features": sorted(
                shared_top
            )
        })


top_feature_overlap = pd.DataFrame(
    overlap_rows
)


print("\n" + "=" * 70)
print("TOP-10 FEATURE OVERLAP")
print("=" * 70)

display(
    top_feature_overlap[
        [
            "representation_a",
            "representation_b",
            "PC",
            "shared_top_features",
            "overlap_fraction"
        ]
    ].round(3)
)


# ------------------------------------------------------------
# 4. Compact summary by PC
# ------------------------------------------------------------

summary_rows = []

for pc in range(
    1,
    N_COMPARE_PCS + 1
):

    subset = pairwise_loading_stability[
        pairwise_loading_stability["PC"] == pc
    ]

    overlap_subset = top_feature_overlap[
        top_feature_overlap["PC"] == pc
    ]

    summary_rows.append({
        "PC": pc,
        "mean_sign_aligned_loading_corr": (
            subset[
                "sign_aligned_loading_corr"
            ].mean()
        ),
        "median_sign_aligned_loading_corr": (
            subset[
                "sign_aligned_loading_corr"
            ].median()
        ),
        "min_sign_aligned_loading_corr": (
            subset[
                "sign_aligned_loading_corr"
            ].min()
        ),
        "mean_absolute_loading_corr": (
            subset[
                "absolute_loading_corr"
            ].mean()
        ),
        "mean_top10_overlap": (
            overlap_subset[
                "overlap_fraction"
            ].mean()
        ),
    })


loading_stability_summary = pd.DataFrame(
    summary_rows
)


print("\n" + "=" * 70)
print("LOADING STABILITY SUMMARY")
print("=" * 70)

display(
    loading_stability_summary.round(3)
)


# ------------------------------------------------------------
# 5. Validation
# ------------------------------------------------------------

assert len(pairwise_loading_stability) == (
    15 * N_COMPARE_PCS
)

assert len(top_feature_overlap) == (
    15 * N_COMPARE_PCS
)

assert np.isfinite(
    pairwise_loading_stability[
        "sign_aligned_loading_corr"
    ]
).all()

assert np.isfinite(
    pairwise_loading_stability[
        "absolute_loading_corr"
    ]
).all()

assert (
    top_feature_overlap[
        "overlap_fraction"
    ].between(0, 1).all()
)


print("\n" + "=" * 70)
print("06D.3 VALIDATION PASSED")
print("=" * 70)

print("""
✓ Shared-feature comparisons performed
✓ PCA sign indeterminacy handled
✓ Signed loading correlations calculated
✓ Absolute loading correlations calculated
✓ Top-10 feature overlap calculated
✓ PC1–PC3 stability summarized

Ready for interpretation.
""")

06D.3 — CROSS-REPRESENTATION LOADING STABILITY

PAIRWISE LOADING CORRELATIONS


,representation_a,representation_b,PC,n_shared_features,raw_loading_corr,sign_aligned_loading_corr,absolute_loading_corr,sign_alignment
0,baseline,distribution_aware,1,50,0.989,0.989,0.977,1
1,baseline,distribution_aware,2,50,0.963,0.963,0.922,1
2,baseline,distribution_aware,3,50,0.925,0.925,0.835,1
3,baseline,yeo_johnson,1,50,0.986,0.986,0.969,1
4,baseline,yeo_johnson,2,50,0.964,0.964,0.924,1
5,baseline,yeo_johnson,3,50,0.874,0.874,0.727,1
6,baseline,block_balanced,1,50,0.853,0.853,0.678,1
7,baseline,block_balanced,2,50,0.922,0.922,0.816,1
8,baseline,block_balanced,3,50,0.819,0.819,0.739,1
9,baseline,ndsi_derived_excluded,1,48,0.999,0.999,0.999,1



TOP-10 FEATURE OVERLAP


,representation_a,representation_b,PC,shared_top_features,overlap_fraction
0,baseline,distribution_aware,1,10,1.0
1,baseline,distribution_aware,2,8,0.8
2,baseline,distribution_aware,3,7,0.7
3,baseline,yeo_johnson,1,10,1.0
4,baseline,yeo_johnson,2,8,0.8
5,baseline,yeo_johnson,3,6,0.6
6,baseline,block_balanced,1,5,0.5
7,baseline,block_balanced,2,7,0.7
8,baseline,block_balanced,3,8,0.8
9,baseline,ndsi_derived_excluded,1,10,1.0



LOADING STABILITY SUMMARY


,PC,mean_sign_aligned_loading_corr,median_sign_aligned_loading_corr,min_sign_aligned_loading_corr,mean_absolute_loading_corr,mean_top10_overlap
0,1,0.947,0.989,0.831,0.884,0.833
1,2,0.956,0.974,0.870,0.903,0.767
2,3,0.857,0.852,0.642,0.749,0.660



06D.3 VALIDATION PASSED

✓ Shared-feature comparisons performed
✓ PCA sign indeterminacy handled
✓ Signed loading correlations calculated
✓ Absolute loading correlations calculated
✓ Top-10 feature overlap calculated
✓ PC1–PC3 stability summarized

Ready for interpretation.



In [4]:
# ============================================================
# 06D.4 — CONSENSUS FEATURE IMPORTANCE ACROSS REPRESENTATIONS
# ============================================================

print("=" * 70)
print("06D.4 — CONSENSUS FEATURE IMPORTANCE")
print("=" * 70)


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

CONSENSUS_PCS = [1, 2, 3]
TOP_K = 10


# ------------------------------------------------------------
# 1. Collect feature-level loading information
# ------------------------------------------------------------

consensus_rows = []

for pc in CONSENSUS_PCS:

    pc_index = pc - 1

    for name in REPRESENTATION_NAMES:

        result = PCA_06D[name]

        features = result["feature_names"]
        loadings = result["loadings"][:, pc_index]

        # Rank by absolute loading
        ranks = (
            pd.Series(
                np.abs(loadings),
                index=features
            )
            .rank(
                ascending=False,
                method="average"
            )
        )

        for feature, loading in zip(
            features,
            loadings
        ):

            consensus_rows.append({
                "PC": pc,
                "representation": name,
                "feature": feature,
                "loading": loading,
                "abs_loading": abs(loading),
                "rank": ranks[feature]
            })


consensus_long = pd.DataFrame(
    consensus_rows
)


# ------------------------------------------------------------
# 2. Focus on features present in all representations
# ------------------------------------------------------------

feature_sets = [
    set(PCA_06D[name]["feature_names"])
    for name in REPRESENTATION_NAMES
]

COMMON_FEATURES = sorted(
    set.intersection(*feature_sets)
)

print("\nCommon features across all six representations:")
print(len(COMMON_FEATURES))


# ------------------------------------------------------------
# 3. Calculate consensus statistics
# ------------------------------------------------------------

consensus_summary_rows = []

for pc in CONSENSUS_PCS:

    subset = consensus_long[
        (
            consensus_long["PC"] == pc
        )
        &
        (
            consensus_long["feature"].isin(
                COMMON_FEATURES
            )
        )
    ]

    for feature in COMMON_FEATURES:

        feature_subset = subset[
            subset["feature"] == feature
        ]

        abs_loadings = (
            feature_subset["abs_loading"]
        )

        ranks = feature_subset["rank"]

        # How many representations place the
        # feature in the top 10?
        top10_count = int(
            np.sum(
                ranks <= TOP_K
            )
        )

        # Mean absolute loading across
        # all six representations.
        mean_abs_loading = (
            abs_loadings.mean()
        )

        median_abs_loading = (
            abs_loadings.median()
        )

        # Mean rank across representations.
        mean_rank = ranks.mean()

        # Sign consistency.
        signs = np.sign(
            feature_subset["loading"]
        )

        positive_fraction = np.mean(
            signs > 0
        )

        negative_fraction = np.mean(
            signs < 0
        )

        consensus_rowsafe = {
            "PC": pc,
            "feature": feature,
            "mean_abs_loading": mean_abs_loading,
            "median_abs_loading": median_abs_loading,
            "mean_rank": mean_rank,
            "top10_count": top10_count,
            "top10_fraction": (
                top10_count
                / len(REPRESENTATION_NAMES)
            ),
            "positive_fraction": positive_fraction,
            "negative_fraction": negative_fraction
        }

        consensus_summary_rows.append(
            consensus_rowsafe
        )


consensus_feature_summary = pd.DataFrame(
    consensus_summary_rows
)


# ------------------------------------------------------------
# 4. Rank consensus features
# ------------------------------------------------------------

consensus_feature_summary[
    "consensus_score"
] = (
    consensus_feature_summary[
        "top10_fraction"
    ]
    * consensus_feature_summary[
        "mean_abs_loading"
    ]
)


# ------------------------------------------------------------
# 5. Display top consensus features
# ------------------------------------------------------------

for pc in CONSENSUS_PCS:

    subset = (
        consensus_feature_summary[
            consensus_feature_summary["PC"] == pc
        ]
        .sort_values(
            [
                "top10_fraction",
                "mean_abs_loading"
            ],
            ascending=False
        )
        .head(15)
    )

    print("\n" + "=" * 70)
    print(f"PC{pc} — CONSENSUS FEATURE IMPORTANCE")
    print("=" * 70)

    display(
        subset[
            [
                "feature",
                "mean_abs_loading",
                "median_abs_loading",
                "mean_rank",
                "top10_count",
                "top10_fraction"
            ]
        ].round(4)
    )


# ------------------------------------------------------------
# 6. Feature-family summary
# ------------------------------------------------------------

def feature_family(feature):

    if feature.startswith("mfcc_"):
        if "within_std" in feature:
            return "MFCC within-window variability"
        return "MFCC means"

    if feature.startswith("spectral_entropy"):
        return "Spectral entropy"

    if feature.startswith("spectral_centroid"):
        return "Spectral centroid"

    if feature.startswith("spectral_bandwidth"):
        return "Spectral bandwidth"

    if feature.startswith("spectral_rolloff"):
        return "Spectral rolloff"

    if feature.startswith("rms_"):
        return "Energy / amplitude"

    if feature.startswith("zcr_"):
        return "Zero-crossing rate"

    if feature.startswith("aci"):
        return "Acoustic indices"

    if feature.startswith("adi"):
        return "Acoustic indices"

    if feature.startswith("bioacoustic"):
        return "Acoustic indices"

    if feature.startswith("ndsi"):
        return "NDSI family"

    if feature.startswith("anthropophony"):
        return "NDSI family"

    if feature.startswith("biophony"):
        return "NDSI family"

    if feature.startswith("temporal_entropy"):
        return "Temporal entropy"

    return "Other"


consensus_feature_summary[
    "feature_family"
] = consensus_feature_summary[
    "feature"
].apply(feature_family)


family_summary = (
    consensus_feature_summary
    .groupby(
        ["PC", "feature_family"],
        as_index=False
    )
    .agg(
        mean_abs_loading=(
            "mean_abs_loading",
            "mean"
        ),
        mean_top10_fraction=(
            "top10_fraction",
            "mean"
        ),
        n_features=(
            "feature",
            "count"
        )
    )
    .sort_values(
        ["PC", "mean_top10_fraction"],
        ascending=[True, False]
    )
)


print("\n" + "=" * 70)
print("FEATURE-FAMILY CONSENSUS")
print("=" * 70)

display(
    family_summary.round(4)
)


# ------------------------------------------------------------
# 7. Validation
# ------------------------------------------------------------

assert len(COMMON_FEATURES) == 46

assert set(
    consensus_feature_summary["feature"]
) == set(COMMON_FEATURES)

assert np.isfinite(
    consensus_feature_summary[
        "mean_abs_loading"
    ]
).all()

assert (
    consensus_feature_summary[
        "top10_count"
    ]
    .between(
        0,
        len(REPRESENTATION_NAMES)
    )
    .all()
)

assert (
    consensus_feature_summary[
        "top10_fraction"
    ]
    .between(0, 1)
    .all()
)


print("\n" + "=" * 70)
print("06D.4 VALIDATION PASSED")
print("=" * 70)

print("""
✓ Consensus calculated using features shared across all representations
✓ PC1–PC3 feature rankings aggregated
✓ Top-10 recurrence quantified
✓ Mean and median loading magnitude calculated
✓ Feature-family structure summarized

Ready for 06D finalization and 06E bootstrap stability.
""")

06D.4 — CONSENSUS FEATURE IMPORTANCE

Common features across all six representations:
46

PC1 — CONSENSUS FEATURE IMPORTANCE


,feature,mean_abs_loading,median_abs_loading,mean_rank,top10_count,top10_fraction
20,mfcc_09_within_std_mean,0.7660,0.8770,2.0000,6,1.0000
26,mfcc_12_within_std_mean,0.7560,0.8636,3.3333,6,1.0000
22,mfcc_10_within_std_mean,0.7502,0.8605,4.3333,6,1.0000
24,mfcc_11_within_std_mean,0.7209,0.8268,8.3333,6,1.0000
28,mfcc_13_within_std_mean,0.7042,0.7993,8.8333,6,1.0000
18,mfcc_08_within_std_mean,0.7595,0.8696,3.6667,5,0.8333
14,mfcc_06_within_std_mean,0.7319,0.8347,6.6667,5,0.8333
12,mfcc_05_within_std_mean,0.7283,0.8324,7.1667,5,0.8333
16,mfcc_07_within_std_mean,0.7268,0.8294,8.1667,5,0.8333
8,mfcc_03_within_std_mean,0.6145,0.6938,11.6667,5,0.8333



PC2 — CONSENSUS FEATURE IMPORTANCE


,feature,mean_abs_loading,median_abs_loading,mean_rank,top10_count,top10_fraction
79,spectral_centroid_mean_mean,0.7718,0.8465,1.1667,6,1.0000
87,spectral_rolloff_mean_mean,0.7407,0.8244,2.6667,6,1.0000
90,zcr_mean_mean,0.7292,0.7774,3.1667,6,1.0000
51,mfcc_02_mean,0.7085,0.8096,4.0000,6,1.0000
85,spectral_entropy_eps_mean,0.5094,0.5697,6.5000,6,1.0000
91,zcr_std_mean,0.4947,0.5206,7.6667,5,0.8333
77,spectral_bandwidth_mean_mean,0.4840,0.5491,8.8333,5,0.8333
47,adi_mean,0.4213,0.4809,13.1667,4,0.6667
81,spectral_entropy_eas_mean,0.4196,0.4776,13.1667,3,0.5000
80,spectral_centroid_std_mean,0.3957,0.4090,12.1667,3,0.5000



PC3 — CONSENSUS FEATURE IMPORTANCE


,feature,mean_abs_loading,median_abs_loading,mean_rank,top10_count,top10_fraction
95,mfcc_01_mean,0.5878,0.6190,2.6667,6,1.0000
132,spectral_entropy_eps_skew_mean,0.3848,0.4359,6.6667,6,1.0000
121,rms_mean_mean,0.5196,0.5376,3.8333,5,0.8333
130,spectral_entropy_eps_kurt_mean,0.3708,0.4116,7.6667,5,0.8333
122,rms_std_mean,0.3650,0.3703,9.6667,5,0.8333
128,spectral_entropy_ecu_mean,0.3768,0.3827,6.8333,4,0.6667
99,mfcc_03_mean,0.3479,0.3991,10.1667,4,0.6667
129,spectral_entropy_ecv_mean,0.3365,0.3324,11.0000,4,0.6667
127,spectral_entropy_eas_mean,0.3116,0.3079,12.6667,4,0.6667
134,spectral_rolloff_std_mean,0.3031,0.3529,15.5000,3,0.5000



FEATURE-FAMILY CONSENSUS


,PC,feature_family,mean_abs_loading,mean_top10_fraction,n_features
3,1,MFCC within-window variability,0.6738,0.7051,13
1,1,Energy / amplitude,0.2619,0.1667,2
4,1,Spectral bandwidth,0.4493,0.0833,2
7,1,Spectral rolloff,0.2202,0.0833,2
0,1,Acoustic indices,0.4427,0.0556,3
2,1,MFCC means,0.1087,0.0000,13
5,1,Spectral centroid,0.1512,0.0000,2
6,1,Spectral entropy,0.2400,0.0000,6
8,1,Temporal entropy,0.1680,0.0000,1
9,1,Zero-crossing rate,0.2022,0.0000,2



06D.4 VALIDATION PASSED

✓ Consensus calculated using features shared across all representations
✓ PC1–PC3 feature rankings aggregated
✓ Top-10 recurrence quantified
✓ Mean and median loading magnitude calculated
✓ Feature-family structure summarized

Ready for 06D finalization and 06E bootstrap stability.

